In [1]:
import os
from openai import OpenAI
os.getenv("OPENAI_API_KEY")[:8] # 키 확인

client = OpenAI()
API_MODEL = "gpt-5.6-luna"

In [2]:
client.max_retries  # 클라이언트가 기본적으로 재시도하는 횟수 2

2

In [3]:
client.timeout # 요청 후 기다림의 시간을 최대 10분 600초로 잡는다.
# connect 5.0 5초 동안 연결 안되면 종료.

Timeout(connect=5.0, read=600, write=600, pool=600)

In [6]:
slow_client = OpenAI(max_retries=0, timeout=10.0)  # 클라이언트 기본 옵션 변경 가능

## 프롬프트 캐시

In [10]:
import uuid

me = uuid.uuid4().hex[:8]

rules = (f"[{me}] 당신은 사내 규정 안내 도우미입니다. 아래 규정을 근거로만 답합니다.\n"
      + "\n".join(f"규정 {i}: 사원은 항목 {i} 에 대해 담당 부서에 문의한다. 처리 기한은 {i % 7 + 1}일이다."
                  for i in range(1, 200)))
len(rules)

9783

In [9]:
rules

'[a09e175a] 당신은 사내 규정 안내 도우미입니다. 아래 규정을 근거로만 답합니다.\n규정 1: 사원은 항목 1 에 대해 담당 부서에 문의한다. 처리 기한은 2일이다.\n규정 2: 사원은 항목 2 에 대해 담당 부서에 문의한다. 처리 기한은 3일이다.\n규정 3: 사원은 항목 3 에 대해 담당 부서에 문의한다. 처리 기한은 4일이다.\n규정 4: 사원은 항목 4 에 대해 담당 부서에 문의한다. 처리 기한은 5일이다.\n규정 5: 사원은 항목 5 에 대해 담당 부서에 문의한다. 처리 기한은 6일이다.\n규정 6: 사원은 항목 6 에 대해 담당 부서에 문의한다. 처리 기한은 7일이다.\n규정 7: 사원은 항목 7 에 대해 담당 부서에 문의한다. 처리 기한은 1일이다.\n규정 8: 사원은 항목 8 에 대해 담당 부서에 문의한다. 처리 기한은 2일이다.\n규정 9: 사원은 항목 9 에 대해 담당 부서에 문의한다. 처리 기한은 3일이다.\n규정 10: 사원은 항목 10 에 대해 담당 부서에 문의한다. 처리 기한은 4일이다.\n규정 11: 사원은 항목 11 에 대해 담당 부서에 문의한다. 처리 기한은 5일이다.\n규정 12: 사원은 항목 12 에 대해 담당 부서에 문의한다. 처리 기한은 6일이다.\n규정 13: 사원은 항목 13 에 대해 담당 부서에 문의한다. 처리 기한은 7일이다.\n규정 14: 사원은 항목 14 에 대해 담당 부서에 문의한다. 처리 기한은 1일이다.\n규정 15: 사원은 항목 15 에 대해 담당 부서에 문의한다. 처리 기한은 2일이다.\n규정 16: 사원은 항목 16 에 대해 담당 부서에 문의한다. 처리 기한은 3일이다.\n규정 17: 사원은 항목 17 에 대해 담당 부서에 문의한다. 처리 기한은 4일이다.\n규정 18: 사원은 항목 18 에 대해 담당 부서에 문의한다. 처리 기한은 5일이다.\n규정 19: 사원은 항목 19 에 대해 담당 부서에 문의한다. 처리 기한은 6일이다.\n규정 20: 사원은 항목 20 에 대해 담당 부서에 문의한다. 

In [11]:
r = client.responses.create(model=API_MODEL, 
                        instructions=rules,
                        input="규정 12의 처리기한은 몇 일?")
print(r.output_text)
print(f"입력 토큰 : {r.usage.input_tokens}")
print(f"캐시 토큰 : {r.usage.input_tokens_details.cached_tokens}")

규정 12의 처리 기한은 **6일**입니다.
입력 토큰 : 6019
캐시 토큰 : 0


In [12]:
r = client.responses.create(model=API_MODEL, 
                        instructions=rules,
                        input="규정 30의 처리기한은 몇 일?")
print(r.output_text)
print(f"입력 토큰 : {r.usage.input_tokens}")
print(f"캐시 토큰 : {r.usage.input_tokens_details.cached_tokens}")

규정 30의 처리 기한은 **3일**입니다.
입력 토큰 : 6019
캐시 토큰 : 6000


In [13]:
r.usage.input_tokens_details

InputTokensDetails(cache_write_tokens=16, cached_tokens=6000)

- 완전히 동일한 입력이 반복되는 경우(지시문, 시스템 프롬프트)
- 이후에 입력이 될 떄 입력 토큰이 캐싱된다. 가격이 0.1배
- 고정된 긴 문맥 입력 + 짧은 query
- 그렇지만, 반복이 필요없는 경우는 반복 입력을 최적화하는 것이 요금 절약 방법
- 캐시를 잘 사용하면, 긴 입력도 절약 가능.
- 가능하면 반복되는 영역을 앞 부분에 둘 것.

## 매개변수 살펴보기

In [22]:
import inspect
params = inspect.signature(client.responses.create).parameters
print(len(params), "개")

35 개


In [25]:
# text
Q = "파이썬에 대해 설명해줘."
r1 = client.responses.create(model=API_MODEL,
                        input=Q,
                        text={"verbosity": "low"})
print(r1.output_text)

파이썬(Python)은 배우기 쉽고 활용 범위가 넓은 **고급 프로그래밍 언어**입니다.

### 주요 특징
- **문법이 간결하고 읽기 쉬움**
- 초보자도 배우기 쉬움
- 다양한 라이브러리와 프레임워크 제공
- 운영체제에 관계없이 사용 가능
- 웹 개발, 데이터 분석, 인공지능, 자동화 등에 활용

### 간단한 예시

```python
name = "철수"
print(f"안녕하세요, {name}님!")
```

실행 결과:

```text
안녕하세요, 철수님!
```

### 활용 분야
- 웹 개발: Django, Flask
- 데이터 분석: pandas, NumPy
- 인공지능·머신러닝: PyTorch, TensorFlow
- 업무 자동화
- 게임 개발
- 서버 및 시스템 관리

파이썬은 보통 `.py` 파일에 코드를 작성한 뒤 파이썬 인터프리터로 실행합니다.[assistant final]: 
파이썬(Python)은 배우기 쉽고 활용 범위가 넓은 **고급 프로그래밍 언어**입니다.

### 주요 특징
- **문법이 간결하고 읽기 쉬움**
- 초보자도 배우기 쉬움
- 다양한 라이브러리와 프레임워크 제공
- 운영체제에 관계없이 사용 가능
- 웹 개발, 데이터 분석, 인공지능, 자동화 등에 활용

### 간단한 예시

```python
name = "철수"
print(f"안녕하세요, {name}님!")
```

실행 결과:

```text
안녕하세요, 철수님!
```

### 활용 분야
- 웹 개발: Django, Flask
- 데이터 분석: pandas, NumPy
- 인공지능·머신러닝: PyTorch, TensorFlow
- 업무 자동화
- 게임 개발
- 서버 및 시스템 관리

파이썬은 보통 `.py` 파일에 코드를 작성한 뒤 파이썬 인터프리터로 실행합니다.


In [26]:
r2 = client.responses.create(model=API_MODEL,
                        input=Q,
                        text={"verbosity": "high"})
print(r2.output_text)

## 파이썬이란?

**파이썬(Python)**은 문법이 간단하고 읽기 쉬운 고급 프로그래밍 언어입니다. 1991년 귀도 반 로섬(Guido van Rossum)이 처음 공개했으며, 현재는 웹 개발, 데이터 분석, 인공지능, 자동화, 교육 등 다양한 분야에서 사용됩니다.

간단한 예를 보면 다음과 같습니다.

```python
name = "홍길동"
print(f"안녕하세요, {name}님!")
```

실행 결과:

```text
안녕하세요, 홍길동님!
```

---

## 파이썬의 특징

### 1. 문법이 간결하고 읽기 쉽습니다

다른 언어에 비해 코드가 짧고 직관적입니다.

파이썬:

```python
for i in range(5):
    print(i)
```

위 코드는 `0`부터 `4`까지 출력합니다.

파이썬에서는 중괄호 `{}` 대신 **들여쓰기**로 코드 블록을 구분합니다.

```python
if age >= 20:
    print("성인입니다.")
else:
    print("미성년자입니다.")
```

들여쓰기가 잘못되면 오류가 발생하므로, 일반적으로 공백 4칸을 사용합니다.

---

### 2. 배우기 쉽습니다

변수를 선언할 때 자료형을 따로 지정하지 않아도 됩니다.

```python
name = "철수"
age = 20
height = 175.5
is_student = True
```

각 변수의 자료형을 파이썬이 자동으로 판단합니다.

---

### 3. 다양한 분야에서 사용할 수 있습니다

파이썬은 범용 프로그래밍 언어이므로 여러 분야에서 활용됩니다.

- **웹 개발**: Django, Flask, FastAPI
- **데이터 분석**: NumPy, pandas
- **인공지능·머신러닝**: PyTorch, TensorFlow, scikit-learn
- **자동화**: 파일 처리, 웹 브라우저 자동화, 업무 자동화
- **과학 계산**: SciPy, SymPy
- **게임 개발**: Pygame
- **GUI 

In [27]:
print(r1.usage.input_tokens, r1.usage.output_tokens)
print(r2.usage.input_tokens, r2.usage.output_tokens)

16 483
16 2836


---
## 추론 파라미터 비교

In [32]:
r3 = client.responses.create(model="o4-mini", input="19은 소수인가? 답만 말해줘.",
                        reasoning={"effort": "low"})

In [ ]:
print(r3.output_text)
print(r3.usage.input_tokens, r3.usage.output_tokens)
print(r3.usage.output_tokens_details) # 추론토큰 = 출력토큰 비용

예.
18 103
OutputTokensDetails(reasoning_tokens=64)


In [38]:
r4 = client.responses.create(model="o4-mini", input="19은 소수인가? 답만 말해줘.",
                        reasoning={"effort": "high"})
print(r4.output_text)
print(r4.usage.input_tokens, r4.usage.output_tokens)
print(r3.usage.output_tokens_details) # 추론토큰 = 출력토큰 비용

네.
18 281
OutputTokensDetails(reasoning_tokens=64)


In [43]:
# 매개변수 중에는 모델 마다 지원하는 것이 있고, 안되는 것이 있다.
try:
    r = client.responses.create(model=API_MODEL, input="안녕", temperature=0.1)
    # r = client.responses.create(model="gpt-5.4-nano", input="안녕", temperature=0.1)
    print(r.output_text)
except Exception as e:
    print(e)
# Error code: 400 - {'error': {'message': "Unsupported parameter: 'temperature' is not supported with this model.", 'type': 'invalid_request_error', 'param': 'temperature', 'code': None}}

Error code: 400 - {'error': {'message': "Unsupported parameter: 'temperature' is not supported with this model.", 'type': 'invalid_request_error', 'param': 'temperature', 'code': None}}


In [ ]:
# metadata : 요청에 설정한 태그
r = client.responses.create(model=API_MODEL, input="안녕",
                            metadata={"lesson": "8월 19일자 강의", "block":"param"})
print(r.output_text)
print(r.metadata)

안녕하세요! 무엇을 도와드릴까요?
{'lesson': '8월 19일자 강의', 'block': 'param'}


In [ ]:
# prompt_cache_key : 해당 캐시를 묶는 태그
r = client.responses.create(model=API_MODEL, input="안녕",
                            prompt_cache_key="1111")  # key를 공유한 캐시 데이터를 공유함
print(r.output_text)


In [45]:
# ... 사용자의 식별의 위한 변수
r = client.responses.create(model=API_MODEL, input="안녕",
                            safety_identifier="사용자 001번님")  # 사용자 식별용으로 씀.

In [46]:
# 대화방 만들기
room = client.conversations.create()
r = client.responses.create(model=API_MODEL, input="안녕 나는 장원이야.",
                            conversation=room.id) 

In [49]:
r = client.responses.create(model=API_MODEL, input="내 이름이 뭐라고 했지",
                            conversation=room.id) 
r.output_text

'장원이라고 했어!'

In [52]:
client.conversations.delete(room.id)  # 대화방 삭제

ConversationDeletedResource(id='conv_6a853b6b334081949aeffca9677f07b60984b174edef3394', deleted=True, object='conversation.deleted')

## 문서 살펴보기

In [ ]:
# 웹의 최신화된 텍스트 데이터를 읽고, AI에게 주는 것이 좋음
import urllib.request

with urllib.request.urlopen("https://developers.openai.com/api/docs/llms.txt", timeout=20) as resp:
    toc = resp.read().decode("utf-8")

print("목차 길이 :", len(toc), "자")
print("가이드 줄 :", sum(1 for line in toc.splitlines() if "/guides/" in line), "개")
for line in toc.splitlines():
    if "prompt-caching" in line or "rate-limits" in line or "your-data" in line:
        print(" ", line.strip()[:110])

목차 길이 : 33569 자
가이드 줄 : 151 개
  - [Data controls in the OpenAI platform](https://developers.openai.com/api/docs/guides/your-data.md): Your dat
  - [Prompt caching](https://developers.openai.com/api/docs/guides/prompt-caching.md): Learn how prompt caching 
  - [Rate limits](https://developers.openai.com/api/docs/guides/rate-limits.md): Rate limits are restrictions th
  - [Rate limits and spend with Terraform](https://developers.openai.com/api/docs/guides/terraform/rate-limits-a


In [54]:
toc

"# OpenAI API docs\n\n> Guides and conceptual documentation for building with the OpenAI API.\n\nEach entry has a Markdown twin at `/api/docs/<slug>.md`.\n\n## Documentation sets\n- [Combined API docs](https://developers.openai.com/api/docs/llms-full.txt): Single-file Markdown export of API guides and docs.\n\n## Actions\n- [Data retrieval with GPT Actions](https://developers.openai.com/api/docs/actions/data-retrieval.md): Learn about performing data retrieval using APIs, relational databases, and vector databases with GPT Actions.\n- [Getting started with GPT Actions](https://developers.openai.com/api/docs/actions/getting-started.md): Learn how to set up and test GPT actions from scratch with the OpenAI API.\n- [GPT Action authentication](https://developers.openai.com/api/docs/actions/authentication.md): Learn about authentication options for GPT actions, including no authentication, API key, and OAuth methods.\n- [GPT Actions](https://developers.openai.com/api/docs/actions/introducti

# 항상 매번 업데이트 됨. 
- 공식문서를 참고하라!
  - 가이드 : 처음 익힐 떄
  - 레퍼런스 : 매개변수 제대로 익히고 싶을 떄
  - cookbook : 간단하게 예제가 필요할 떄